In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# 1. MOCK DATA GENERATION based on the schema
def generate_advanced_mock_data(n=200):
    np.random.seed(42)
    data = {
        'avg_lumen': np.random.uniform(200, 2000, n),
        'avg_celsius': np.random.uniform(18, 26, n),
        'avg_ppm': np.random.uniform(400, 2000, n),
        'avg_ml': np.random.uniform(0, 1000, n),
        'sleep_duration': [f"{np.random.randint(4, 10)} hours {np.random.randint(0, 60)} minutes" for _ in range(n)],
        'eco_code': np.random.choice(['A00', 'B20', 'C50', 'D30', 'E60'], n),
        'total_ply': np.random.randint(20, 120, n),
        'opening_ply': np.random.randint(5, 25, n),
        'player_move_count': np.random.randint(10, 60, n),
        'opponent_move_count': np.random.randint(10, 60, n),
        'time_control': np.random.choice(['blitz', 'rapid', 'classical'], n),
        'is_time_increase': np.random.choice([True, False], n),
        'time_increase_sec': np.random.randint(0, 30, n),
        'is_berserk': np.random.choice([True, False], n),
        'duration_min': np.random.randint(1, 30, n),
        'user_rating': np.random.randint(1200, 2800, n),
        'opp_rating': np.random.randint(1200, 2800, n),
        'rating_diff': np.random.randint(-300, 300, n),
        'is_player_piece_black': np.random.choice([True, False], n),
        'termination_type': np.random.choice(['Normal', 'Time Forfeit', 'Resignation'], n),
        'result': np.random.choice(['Win', 'Loss', 'Draw'], n, p=[0.45, 0.45, 0.1]),
        'player_opening_win_rate': np.random.uniform(0.3, 0.7, n),
        'player_opening_game_count': np.random.randint(1, 1000, n)
    }
    return pd.DataFrame(data)

df = generate_advanced_mock_data()

# 2. PREPROCESSING FUNCTIONS
def preprocess_chess_data(df):
    # Convert sleep_duration (INTERVAL) to float hours
    # This is a simple regex/split approach for the mock format
    def parse_sleep(s):
        parts = s.split()
        return float(parts[0]) + float(parts[2])/60.0
    
    df_proc = df.copy()
    df_proc['sleep_hours'] = df_proc['sleep_duration'].apply(parse_sleep)
    
    # Target: 1 for Win, 0 for everything else
    df_proc['target'] = (df_proc['result'] == 'Win').astype(int)
    
    return df_proc

df_cleaned = preprocess_chess_data(df)

# Define feature groups
env_cols = ['avg_lumen', 'avg_celsius', 'avg_ppm', 'avg_ml']
game_cols = [
    'rating_diff', 'total_ply', 'opening_ply', 'player_move_count', 
    'opponent_move_count', 'duration_min', 'player_opening_win_rate', 
    'player_opening_game_count', 'sleep_hours'
]
cat_cols = ['eco_code', 'time_control', 'is_berserk', 'is_player_piece_black']

# 3. DUAL-CLUSTERING ARCHITECTURE
# Split Data
X = df_cleaned[env_cols + game_cols + cat_cols]
y = df_cleaned['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Fit Scalers & Clusters on training data
scaler_env = StandardScaler().fit(X_train[env_cols])
scaler_game = StandardScaler().fit(X_train[game_cols])

# Cluster Environment (Room state)
env_train_scaled = scaler_env.transform(X_train[env_cols])
kmeans_env = KMeans(n_clusters=3, random_state=42, n_init=10).fit(env_train_scaled)

# Cluster Game Parameters (Performance state)
game_train_scaled = scaler_game.transform(X_train[game_cols])
kmeans_game = KMeans(n_clusters=4, random_state=42, n_init=10).fit(game_train_scaled)

# Add Cluster Labels to features
X_train_final = X_train.copy()
X_train_final['env_cluster'] = kmeans_env.labels_
X_train_final['game_cluster'] = kmeans_game.labels_

# Apply to Test Set
X_test_final = X_test.copy()
X_test_final['env_cluster'] = kmeans_env.predict(scaler_env.transform(X_test[env_cols]))
X_test_final['game_cluster'] = kmeans_game.predict(scaler_game.transform(X_test[game_cols]))

# One-Hot Encode Categorical features + Clusters
# Note: Treat cluster IDs as categorical
cat_features_to_encode = cat_cols + ['env_cluster', 'game_cluster']
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features_to_encode)
    ],
    remainder='passthrough'
)

# 4. FINAL PIPELINE
clf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

clf_pipeline.fit(X_train_final, y_train)
y_pred = clf_pipeline.predict(X_test_final)

print("Classification Report for the Reworked Model:")
print(classification_report(y_test, y_pred))

# Save the logic to CSV for user inspection
df_cleaned.to_csv('reworked_mock_data.csv', index=False)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# 1. DATA CLEANING
def clean_dataset(df):
    # Convert sleep_duration (INTERVAL) to float hours for mathematical scaling
    # Assuming standard ISO duration or "H hours M minutes" string
    df['sleep_hours'] = pd.to_timedelta(df['sleep_duration']).dt.total_seconds() / 3600.0
    
    # Define the Target (1 = Win, 0 = Loss/Draw)
    df['target'] = (df['result'] == 'Win').astype(int)
    return df

# 2. FEATURE SELECTION
env_cols = ['avg_lumen', 'avg_celsius', 'avg_ppm', 'avg_ml']
game_cols = [
    'rating_diff', 'total_ply', 'opening_ply', 'player_move_count', 
    'duration_min', 'player_opening_win_rate', 'sleep_hours'
]
cat_cols = ['eco_code', 'time_control', 'is_berserk', 'is_player_piece_black']

# 3. DUAL-CLUSTERING LOGIC
def apply_dual_clustering(X_train, X_test):
    # Scale and Cluster Environment
    scaler_env = StandardScaler()
    env_train_scaled = scaler_env.fit_transform(X_train[env_cols])
    km_env = KMeans(n_clusters=3, random_state=42).fit(env_train_scaled)
    
    # Scale and Cluster Game/Player State
    scaler_game = StandardScaler()
    game_train_scaled = scaler_game.fit_transform(X_train[game_cols])
    km_game = KMeans(n_clusters=4, random_state=42).fit(game_train_scaled)
    
    # Assign Clusters as new categorical features
    for df, s_env, s_game in [(X_train, scaler_env, scaler_game), (X_test, scaler_env, scaler_game)]:
        df['env_cluster'] = km_env.predict(s_env.transform(df[env_cols]))
        df['game_cluster'] = km_game.predict(s_game.transform(df[game_cols]))
    
    return X_train, X_test

# 4. TRAINING PIPELINE
# One-hot encode the categorical variables and the newly created clusters
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols + ['env_cluster', 'game_cluster'])
    ],
    remainder='passthrough'
)

model_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42))
])

# Usage:
# df = pd.read_sql("SELECT * FROM games", conn)
# df = clean_dataset(df)
# X_train, X_test, y_train, y_test = train_test_split(df.drop('target', axis=1), df['target'])
# X_train, X_test = apply_dual_clustering(X_train, X_test)
# model_pipeline.fit(X_train, y_train)